# 📈 Regressão Linear — Fundamentos e Métricas de Validação
**FIAP Postech — IA para Devs | Fase 1 · Machine Learning**

---

## O que você vai aprender
- Diferença entre Regressão e Classificação
- Regressão Linear Simples e Múltipla
- Feature Scaling e sua influência nos coeficientes
- Métricas: MAE, MSE, RMSE, R²
- Diagnóstico de overfitting com train/test split e cross-validation
- Regularização: Ridge (L2) e Lasso (L1)

---
> **Regressão vs Classificação**  
> Classificação → saída é uma **categoria** (maligno/benigno)  
> Regressão → saída é um **número contínuo** (preço, temperatura, salário)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('✅ Imports OK')

ModuleNotFoundError: No module named 'sklearn'

---
## 1. Dataset — California Housing
Prever o **preço médio de imóveis** (em $100k) com base em características do bairro.

In [ ]:
raw = fetch_california_housing()
df = pd.DataFrame(raw.data, columns=raw.feature_names)
df['Price'] = raw.target  # preço em $100k

print('Shape:', df.shape)
print('\nFeatures:')
for name, desc in zip(raw.feature_names, raw.feature_names):
    print(f'  {name}')
print('\nTarget: MedHouseVal (mediana do valor dos imóveis em $100k)')
df.head()

In [ ]:
df.describe().T

In [ ]:
# Distribuição do target
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df['Price'], bins=50, color='#3498db', edgecolor='white')
axes[0].set_title('Distribuição do Preço', fontweight='bold')
axes[0].set_xlabel('Preço ($100k)')
axes[0].set_ylabel('Frequência')

axes[1].boxplot(df['Price'], vert=False, patch_artist=True,
                boxprops=dict(facecolor='#3498db', alpha=0.7))
axes[1].set_title('Boxplot do Preço', fontweight='bold')
axes[1].set_xlabel('Preço ($100k)')

plt.tight_layout()
plt.show()

In [ ]:
# Correlação das features com o target
corr = df.corr()['Price'].drop('Price').sort_values()

plt.figure(figsize=(8, 5))
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in corr.values]
corr.plot(kind='barh', color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Correlação das Features com o Preço', fontweight='bold')
plt.xlabel('Correlação de Pearson')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: MedInc vs Price (maior correlação)
plt.figure(figsize=(8, 5))
plt.scatter(df['MedInc'], df['Price'], alpha=0.15, color='#3498db', s=10)
plt.xlabel('Renda Média (MedInc)')
plt.ylabel('Preço ($100k)')
plt.title('Renda Média vs Preço do Imóvel', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 2. Regressão Linear Simples
Primeiro com apenas 1 feature (MedInc) para visualizar a reta de regressão.

In [ ]:
X_simple = df[['MedInc']]
y = df['Price']

X_tr, X_te, y_tr, y_te = train_test_split(X_simple, y, test_size=0.2, random_state=42)

lr_simple = LinearRegression()
lr_simple.fit(X_tr, y_tr)
y_pred_simple = lr_simple.predict(X_te)

print(f'Intercepto (b₀): {lr_simple.intercept_:.4f}')
print(f'Coeficiente (b₁): {lr_simple.coef_[0]:.4f}')
print(f'\nEquação: Preço = {lr_simple.intercept_:.2f} + {lr_simple.coef_[0]:.2f} × MedInc')
print(f'\nInterpretação: para cada unidade de MedInc, o preço aumenta ${lr_simple.coef_[0]*100:.0f}')

In [ ]:
# Visualizando a reta de regressão
x_line = np.linspace(X_simple['MedInc'].min(), X_simple['MedInc'].max(), 100).reshape(-1, 1)
y_line = lr_simple.predict(x_line)

plt.figure(figsize=(9, 5))
plt.scatter(X_te, y_te, alpha=0.2, color='#3498db', s=10, label='Dados reais')
plt.plot(x_line, y_line, color='#e74c3c', lw=2.5, label='Reta de regressão')
plt.xlabel('Renda Média (MedInc)')
plt.ylabel('Preço ($100k)')
plt.title('Regressão Linear Simples — MedInc → Preço', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

---
## 3. Métricas de Avaliação

| Métrica | Fórmula | Interpretação |
|---|---|---|
| **MAE** | mean(\|y - ŷ\|) | Erro médio absoluto — mesma unidade do target |
| **MSE** | mean((y - ŷ)²) | Penaliza erros grandes — sensível a outliers |
| **RMSE** | √MSE | MSE na unidade original — mais interpretável |
| **R²** | 1 - SS_res/SS_tot | % da variância explicada — 1.0 = perfeito, 0 = média |

In [ ]:
def avaliar_modelo(nome, y_real, y_pred):
    mae  = mean_absolute_error(y_real, y_pred)
    mse  = mean_squared_error(y_real, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_real, y_pred)
    print(f'\n📊 {nome}')
    print(f'   MAE:  {mae:.4f}  → erro médio de ${mae*100:.0f}')
    print(f'   MSE:  {mse:.4f}')
    print(f'   RMSE: {rmse:.4f} → desvio típico de ${rmse*100:.0f}')
    print(f'   R²:   {r2:.4f}  → modelo explica {r2*100:.1f}% da variância')
    return {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R²': r2}

m_simple = avaliar_modelo('Regressão Simples (1 feature)', y_te, y_pred_simple)

In [ ]:
# Gráfico Resíduo — diagnóstico do modelo
residuos = y_te - y_pred_simple

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(y_pred_simple, residuos, alpha=0.2, color='#9b59b6', s=10)
axes[0].axhline(0, color='red', linestyle='--', lw=1.5)
axes[0].set_xlabel('Predito')
axes[0].set_ylabel('Resíduo (real - predito)')
axes[0].set_title('Resíduos vs Predito', fontweight='bold')

axes[1].hist(residuos, bins=50, color='#9b59b6', edgecolor='white')
axes[1].set_xlabel('Resíduo')
axes[1].set_title('Distribuição dos Resíduos\n(ideal: normal centrada em 0)', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 4. Regressão Linear Múltipla — Todas as Features

In [ ]:
X = df.drop('Price', axis=1)
y = df['Price']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# Pipeline com StandardScaler
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

pipe_lr.fit(X_tr, y_tr)
y_pred_mult = pipe_lr.predict(X_te)

m_mult = avaliar_modelo('Regressão Múltipla (8 features)', y_te, y_pred_mult)

In [ ]:
# Coeficientes padronizados (após scaling)
coefs = pd.Series(
    pipe_lr.named_steps['model'].coef_,
    index=X.columns
).sort_values()

plt.figure(figsize=(8, 5))
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in coefs.values]
coefs.plot(kind='barh', color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Coeficientes Padronizados — Regressão Múltipla', fontweight='bold')
plt.xlabel('Impacto no Preço (em desvios-padrão)')
plt.tight_layout()
plt.show()

print('\n💡 Coeficiente positivo → feature aumenta o preço')
print('   Coeficiente negativo → feature diminui o preço')
print('   Magnitude → quanto maior, mais impacto')

In [ ]:
# Real vs Predito
plt.figure(figsize=(8, 6))
plt.scatter(y_te, y_pred_mult, alpha=0.2, color='#3498db', s=10)
lims = [min(y_te.min(), y_pred_mult.min()), max(y_te.max(), y_pred_mult.max())]
plt.plot(lims, lims, 'r--', lw=2, label='Predição perfeita')
plt.xlabel('Valor Real')
plt.ylabel('Valor Predito')
plt.title(f'Real vs Predito — R²={r2_score(y_te, y_pred_mult):.3f}', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

---
## 5. Validação Cruzada (Cross-Validation)

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores_r2   = cross_val_score(pipe_lr, X, y, cv=kf, scoring='r2')
scores_rmse = np.sqrt(-cross_val_score(pipe_lr, X, y, cv=kf, scoring='neg_mean_squared_error'))

print('📊 Cross-Validation 5-Fold — Regressão Linear Múltipla')
print(f'   R²   por fold: {[f"{s:.3f}" for s in scores_r2]}')
print(f'   R²   mean ± std: {scores_r2.mean():.3f} ± {scores_r2.std():.3f}')
print(f'\n   RMSE por fold: {[f"{s:.3f}" for s in scores_rmse]}')
print(f'   RMSE mean ± std: {scores_rmse.mean():.3f} ± {scores_rmse.std():.3f}')

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fold_labels = [f'Fold {i+1}' for i in range(5)]

axes[0].bar(fold_labels, scores_r2, color='#3498db', edgecolor='white')
axes[0].axhline(scores_r2.mean(), color='red', linestyle='--', label=f'Média={scores_r2.mean():.3f}')
axes[0].set_title('R² por Fold', fontweight='bold')
axes[0].legend()
axes[0].set_ylim(0.5, 0.75)

axes[1].bar(fold_labels, scores_rmse, color='#e67e22', edgecolor='white')
axes[1].axhline(scores_rmse.mean(), color='red', linestyle='--', label=f'Média={scores_rmse.mean():.3f}')
axes[1].set_title('RMSE por Fold', fontweight='bold')
axes[1].legend()

plt.suptitle('Cross-Validation 5-Fold', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Regularização — Ridge (L2) e Lasso (L1)

Quando o modelo overfita (decora o treino, vai mal no teste), adicionamos **penalização nos coeficientes**.

| Técnica | Penalização | Efeito |
|---|---|---|
| **Ridge (L2)** | soma dos quadrados dos coefs | Reduz todos os coeficientes |
| **Lasso (L1)** | soma dos valores absolutos | Zera coeficientes (seleção de feature) |

O parâmetro **alpha (α)** controla a intensidade da regularização.

In [ ]:
modelos_reg = {
    'Linear (sem regularização)': Pipeline([('sc', StandardScaler()), ('m', LinearRegression())]),
    'Ridge α=0.1':  Pipeline([('sc', StandardScaler()), ('m', Ridge(alpha=0.1))]),
    'Ridge α=1.0':  Pipeline([('sc', StandardScaler()), ('m', Ridge(alpha=1.0))]),
    'Ridge α=10':   Pipeline([('sc', StandardScaler()), ('m', Ridge(alpha=10))]),
    'Lasso α=0.01': Pipeline([('sc', StandardScaler()), ('m', Lasso(alpha=0.01))]),
    'Lasso α=0.1':  Pipeline([('sc', StandardScaler()), ('m', Lasso(alpha=0.1))]),
}

resultados = {}
print('Modelo                           | Train R² | Test R²  | RMSE')
print('-' * 65)

for nome, pipe in modelos_reg.items():
    pipe.fit(X_tr, y_tr)
    r2_train = r2_score(y_tr, pipe.predict(X_tr))
    r2_test  = r2_score(y_te, pipe.predict(X_te))
    rmse     = np.sqrt(mean_squared_error(y_te, pipe.predict(X_te)))
    resultados[nome] = {'Train R²': r2_train, 'Test R²': r2_test, 'RMSE': rmse}
    print(f'{nome:<32} | {r2_train:.4f}   | {r2_test:.4f}   | {rmse:.4f}')

In [ ]:
# Efeito do alpha nos coeficientes Lasso (seleção de feature)
alphas = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0]
coef_matrix = []

for a in alphas:
    pipe = Pipeline([('sc', StandardScaler()), ('m', Lasso(alpha=a))])
    pipe.fit(X_tr, y_tr)
    coef_matrix.append(pipe.named_steps['m'].coef_)

coef_df = pd.DataFrame(coef_matrix, index=alphas, columns=X.columns)

plt.figure(figsize=(11, 5))
for col in coef_df.columns:
    plt.plot(alphas, coef_df[col], marker='o', label=col)

plt.xscale('log')
plt.axhline(0, color='black', lw=0.8)
plt.xlabel('Alpha (escala log)')
plt.ylabel('Coeficiente')
plt.title('Lasso — Coeficientes conforme alpha aumenta\n(coefs chegam a zero = feature eliminada)', fontweight='bold')
plt.legend(loc='upper right', fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

---
## 7. Resumo

| Modelo | Quando usar |
|---|---|
| Regressão Linear Simples | 1 feature, relação linear óbvia |
| Regressão Linear Múltipla | várias features, sem multicolinearidade severa |
| Ridge | Quando há multicolinearidade (features correlacionadas) |
| Lasso | Quando você quer seleção automática de features |

**Métricas a reportar sempre:** MAE, RMSE e R². O RMSE pune outliers, o MAE é mais robusto — relate os dois.